# IO y Encodings en Pandas

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/13_pandas_io/code/01_io_y_encodings.ipynb)

Leer datos es el primer paso de cualquier pipeline. Pandas hace que el caso feliz sea trivial, pero en producción los datos rara vez llegan limpios: separadores raros, encodings incorrectos, JSON anidado, columnas que no necesitas. Este notebook cubre los casos que sí te vas a encontrar.

In [1]:
# Ejecutar solo si no tienes las librerías instaladas
# En Colab, ejecuta esta celda primero
%pip install pandas==2.2.3 numpy==1.26.4 pyarrow==17.0.0 chardet==5.2.0 charset-normalizer==3.3.2 openpyxl==3.1.5

In [2]:
import io
import json
import chardet
import numpy as np
import pandas as pd
from charset_normalizer import from_bytes

---

## Sección 1: CSV — más allá del caso básico

### El caso feliz

`pd.read_csv()` funciona perfectamente cuando el archivo sigue la convención: comas como separador, encabezado en la primera fila, strings como UTF-8. Pero eso no siempre pasa.

In [3]:
# io.StringIO simula un archivo en memoria — no necesitas escribir nada al disco
csv_simple = """
nombre,edad,ciudad
Ana,28,CDMX
Luis,34,Guadalajara
María,22,Monterrey
""".strip()

df = pd.read_csv(io.StringIO(csv_simple))
df

,nombre,edad,ciudad
0,Ana,28,CDMX
1,Luis,34,Guadalajara
2,María,22,Monterrey


### Separadores: no todo es una coma

El parámetro `sep=` (o `delimiter=`) controla qué carácter separa columnas. Los más comunes son `;` (Europa, Excel en español), `\t` (TSV) y `|` (logs, bases de datos).

In [4]:
# CSV con punto y coma — muy común en archivos exportados desde Excel en español
csv_puntoycoma = "nombre;sueldo;departamento\nAna;45000;Ingeniería\nLuis;38000;Marketing"

df_sc = pd.read_csv(io.StringIO(csv_puntoycoma), sep=";")
df_sc

,nombre,sueldo,departamento
0,Ana,45000,Ingeniería
1,Luis,38000,Marketing


In [5]:
# TSV (Tab-Separated Values) — común en exports de bases de datos
csv_tsv = "id\tnombre\tvalor\n1\tProducto A\t99.9\n2\tProducto B\t149.9"

df_tsv = pd.read_csv(io.StringIO(csv_tsv), sep="\t")
df_tsv

,id,nombre,valor
0,1,Producto A,99.9
1,2,Producto B,149.9


In [6]:
# Pipe-separated — frecuente en logs y sistemas legacy
csv_pipe = "fecha|evento|usuario\n2026-01-01|login|ana@itam.mx\n2026-01-01|logout|luis@itam.mx"

df_pipe = pd.read_csv(io.StringIO(csv_pipe), sep="|")
df_pipe

,fecha,evento,usuario
0,2026-01-01,login,ana@itam.mx
1,2026-01-01,logout,luis@itam.mx


💡 **Prueba esto:** cambia `sep="|"` por `sep=","` en el último ejemplo y observa qué pasa. Pandas leerá todo como una sola columna.

### Archivos sin header o con metadata arriba

Muchos sistemas exportan archivos con líneas de metadata antes de los datos, o sin encabezado. Los parámetros `header=`, `names=` y `skiprows=` manejan estos casos.

In [7]:
# Archivo con líneas de metadata antes de los datos reales
csv_con_metadata = """
# Reporte mensual de ventas
# Generado: 2026-03-01
# Sistema: ERP v2.1
producto,cantidad,precio
Laptop,5,22000
Monitor,12,8500
Teclado,30,1200
""".strip()

# skiprows salta las primeras N filas antes de buscar el header
df_meta = pd.read_csv(io.StringIO(csv_con_metadata), skiprows=3)
df_meta

,producto,cantidad,precio
0,Laptop,5,22000
1,Monitor,12,8500
2,Teclado,30,1200


In [8]:
# Archivo sin encabezado — pandas asignaría 0, 1, 2... por defecto
csv_sin_header = "Ana,28,CDMX\nLuis,34,Guadalajara\nMaría,22,Monterrey"

# header=None le dice que no hay fila de encabezado
# names= asigna los nombres que tú quieras
df_sh = pd.read_csv(
    io.StringIO(csv_sin_header),
    header=None,
    names=["nombre", "edad", "ciudad"]
)
df_sh

,nombre,edad,ciudad
0,Ana,28,CDMX
1,Luis,34,Guadalajara
2,María,22,Monterrey


💡 **Prueba esto:** quita `header=None` del ejemplo anterior. ¿Qué pasa con la primera fila?

### `usecols=` — leer solo lo que necesitas

Si tienes un CSV con 50 columnas pero solo necesitas 3, leer todo desperdicia memoria. `usecols=` es tu amigo para performance, especialmente con archivos grandes.

In [9]:
csv_muchas_cols = """
id,nombre,apellido,edad,ciudad,estado,pais,telefono,email,salario,departamento
1,Ana,García,28,CDMX,CDMX,MX,5512345678,ana@co.mx,45000,Ingeniería
2,Luis,Martínez,34,GDL,JAL,MX,3312345678,luis@co.mx,38000,Marketing
3,María,López,22,MTY,NL,MX,8112345678,maria@co.mx,52000,Finanzas
""".strip()

# Solo las columnas que realmente vamos a usar — pandas ni siquiera parsea las demás
df_cols = pd.read_csv(
    io.StringIO(csv_muchas_cols),
    usecols=["nombre", "ciudad", "salario"]
)
df_cols

,nombre,ciudad,salario
0,Ana,CDMX,45000
1,Luis,GDL,38000
2,María,MTY,52000


In [10]:
# También puedes seleccionar por posición (índice de columna)
df_cols_idx = pd.read_csv(
    io.StringIO(csv_muchas_cols),
    usecols=[0, 1, 9]  # id, nombre, salario
)
df_cols_idx

,id,nombre,salario
0,1,Ana,45000
1,2,Luis,38000
2,3,María,52000


### `dtype=` — no dejes que pandas infiera los tipos

Pandas infiere tipos al leer, pero esa inferencia puede salir mal. El bug clásico: una columna de enteros que tiene al menos un `NaN` se convierte automáticamente a `float64` porque Python int no acepta nulos.

In [11]:
# El bug: código postal como entero, pero hay un NaN
csv_cp = "nombre,codigo_postal\nAna,06600\nLuis,\nMaría,11000"

df_bug = pd.read_csv(io.StringIO(csv_cp))
print(df_bug.dtypes)
print()
df_bug

nombre            object
codigo_postal    float64
dtype: object



,nombre,codigo_postal
0,Ana,6600.0
1,Luis,NaN
2,María,11000.0


In [12]:
# El problema: 06600 se convierte a 6600.0 — perdiste el cero inicial
# Solución: forzar dtype=str para mantener el valor original
df_ok = pd.read_csv(
    io.StringIO(csv_cp),
    dtype={"codigo_postal": str}
)
print(df_ok.dtypes)
print()
df_ok

nombre           object
codigo_postal    object
dtype: object



,nombre,codigo_postal
0,Ana,06600
1,Luis,NaN
2,María,11000


In [13]:
# Puedes especificar tipos para múltiples columnas a la vez
csv_tipos = "id,precio,categoria\n1,29.99,A\n2,149.50,B\n3,9.99,A"

df_tipos = pd.read_csv(
    io.StringIO(csv_tipos),
    dtype={"id": np.int32, "precio": np.float32, "categoria": "category"}
)
print(df_tipos.dtypes)
df_tipos

id              int32
precio        float32
categoria    category
dtype: object


,id,precio,categoria
0,1,29.99,A
1,2,149.50,B
2,3,9.99,A


💡 **Prueba esto:** agrega `dtype={"id": np.int8}` a un CSV donde id=300. ¿Qué error obtienes? Los tipos enteros tienen límites de tamaño.

### `na_values=` — valores nulos personalizados

Pandas reconoce varios strings como `NaN` por defecto (`""`, `"NA"`, `"NaN"`, `"null"`, `"None"`), pero en datos reales aparecen convenciones propias: `"."`, `"-"`, `"99999"`, `"N/A"`, `"ND"`.

In [14]:
# Dataset con varias convenciones de valores faltantes
csv_nulos = "nombre,edad,ingreso,score\nAna,28,45000,8.5\nLuis,.,38000,N/A\nMaría,22,99999,-\nCarlos,35,52000,ND"

# Sin na_values: pandas solo detecta los nulos estándar
df_sin_na = pd.read_csv(io.StringIO(csv_nulos))
print("Sin na_values:")
print(df_sin_na)
print()

# Con na_values: le decimos exactamente qué strings representan nulos
df_con_na = pd.read_csv(
    io.StringIO(csv_nulos),
    na_values=[".", "-", "99999", "N/A", "ND"]
)
print("Con na_values:")
print(df_con_na)

Sin na_values:
   nombre edad  ingreso score
0     Ana   28    45000   8.5
1    Luis    .    38000   NaN
2   María   22    99999     -
3  Carlos   35    52000    ND

Con na_values:
   nombre  edad  ingreso  score
0     Ana  28.0  45000.0    8.5
1    Luis   NaN  38000.0    NaN
2   María  22.0      NaN    NaN
3  Carlos  35.0  52000.0    NaN


### `parse_dates=` — fechas al leer

En lugar de leer fechas como strings y luego convertirlas con `pd.to_datetime()`, puedes pedirle a pandas que lo haga directamente al leer.

In [15]:
csv_fechas = "evento,fecha_inicio,fecha_fin\nConferencia,2026-03-15,2026-03-17\nTaller,2026-04-01,2026-04-02"

df_fechas = pd.read_csv(
    io.StringIO(csv_fechas),
    parse_dates=["fecha_inicio", "fecha_fin"]  # lista de columnas a parsear como fechas
)
print(df_fechas.dtypes)
print()
df_fechas

evento                  object
fecha_inicio    datetime64[ns]
fecha_fin       datetime64[ns]
dtype: object



,evento,fecha_inicio,fecha_fin
0,Conferencia,2026-03-15,2026-03-17
1,Taller,2026-04-01,2026-04-02


### Ejercicio 1

Tienes el siguiente CSV (mal formateado como suelen venir de Excel en México):

```
# Reporte Q1 2026
producto;ventas;precio_unitario;fecha_venta
Laptop;5;22000;15/01/2026
Monitor;.;8500;20/01/2026
Teclado;30;ND;25/01/2026
```

Lee este CSV de forma que:
- Salte la línea de metadata
- Use `;` como separador
- Trate `.` y `ND` como nulos
- Solo lea las columnas `producto`, `ventas` y `precio_unitario`

In [29]:
csv_ejercicio1 = """
# Reporte Q1 2026
producto;ventas;precio_unitario;fecha_venta
Laptop;5;22000;15/01/2026
Monitor;.;8500;20/01/2026
Teclado;30;ND;25/01/2026
""".strip()

# Tu solución aquí
df_ej1 = pd.read_csv(
    io.StringIO(csv_ejercicio1),
    skiprows=1,
    sep=";",
    na_values=[".", "ND"],
    usecols=["producto", "ventas", "precio_unitario"]
)
df_ej1

,producto,ventas,precio_unitario
0,Laptop,5.0,22000.0
1,Monitor,NaN,8500.0
2,Teclado,30.0,NaN


<details>
<summary>Ver solución</summary>

```python
df_ej1 = pd.read_csv(
    io.StringIO(csv_ejercicio1),
    skiprows=1,
    sep=";",
    na_values=[".", "ND"],
    usecols=["producto", "ventas", "precio_unitario"]
)
```
</details>

---

## Sección 2: Encodings y Mojibake

### ¿Qué es un encoding?

Un archivo de texto es, en el fondo, una secuencia de bytes. Un **encoding** es el mapa que dice cómo traducir esos bytes a caracteres legibles. Si el archivo se guardó con un encoding y lo lees con otro, obtienes **mojibake** — esa basura de caracteres como `Ã¡` en lugar de `á`.

Los encodings más comunes en datos latinoamericanos:

| Encoding | Alias | Contexto |
|---|---|---|
| `utf-8` | — | Estándar universal, soporta todos los idiomas |
| `latin-1` | `iso-8859-1` | Europa occidental, muy común en archivos viejos |
| `cp1252` | `windows-1252` | Windows en América/Europa, similar a latin-1 |
| `utf-8-sig` | — | UTF-8 con BOM (Excel en Windows lo genera) |

### Demostración de Mojibake

El mojibake ocurre cuando un archivo guardado en encoding A se lee como si fuera encoding B. Vamos a replicarlo con bytes reales.

In [17]:
# String original con acentos — texto real en español
texto_original = "Información de configuración: ñoño, güero, Ángel"
print("Original:", texto_original)

# Alguien guardó el archivo con latin-1 (encoding antiguo de Windows/Linux)
bytes_latin1 = texto_original.encode("latin-1")
print("\nBytes (latin-1):", bytes_latin1[:30], "...")

Original: Información de configuración: ñoño, güero, Ángel

Bytes (latin-1): b'Informaci\xf3n de configuraci\xf3n: ' ...


In [18]:
# El problema: otro sistema lee esos bytes asumiendo que son UTF-8
# UTF-8 espera secuencias de bytes distintas para caracteres especiales
try:
    texto_mal = bytes_latin1.decode("utf-8")
    print("Leído como UTF-8:", texto_mal)
except UnicodeDecodeError as e:
    print(f"Error al decodificar como UTF-8: {e}")
    print("\nCon errors='replace':")
    texto_mal = bytes_latin1.decode("utf-8", errors="replace")
    print(texto_mal)

Error al decodificar como UTF-8: 'utf-8' codec can't decode byte 0xf3 in position 9: invalid continuation byte

Con errors='replace':
Informaci�n de configuraci�n: �o�o, g�ero, �ngel


In [19]:
# Mojibake "exitoso": latin-1 codificado, leído como cp1252
# cp1252 puede leer todos los bytes de latin-1 sin error, pero interpreta diferente
bytes_mojibake = "café con ñ".encode("latin-1")

mojibake = bytes_mojibake.decode("cp500", errors="replace")  # encoding distinto
print("Mojibake:", mojibake)
print()

# La solución: decodificar con el encoding correcto
correcto = bytes_mojibake.decode("latin-1")
print("Correcto:", correcto)

Mojibake: Ä/ÃZÄ?>1

Correcto: café con ñ


### Leyendo CSV con encoding específico

In [20]:
# Simular un CSV guardado en latin-1
csv_content = "nombre,ciudad\nJosé,Mérida\nÁngel,Querétaro\nñoño,Campeche"

# Guardarlo como bytes latin-1 (lo que haría un sistema Windows antiguo)
csv_bytes_latin1 = csv_content.encode("latin-1")

# Leer con el encoding correcto
df_lat = pd.read_csv(io.BytesIO(csv_bytes_latin1), encoding="latin-1")
print("Con encoding='latin-1':")
print(df_lat)
print()

Con encoding='latin-1':
  nombre     ciudad
0   José     Mérida
1  Ángel  Querétaro
2   ñoño   Campeche



In [21]:
# ¿Qué pasa si usas el encoding incorrecto?
try:
    df_mal = pd.read_csv(io.BytesIO(csv_bytes_latin1), encoding="utf-8")
    print("Con encoding='utf-8' (incorrecto):")
    print(df_mal)
except UnicodeDecodeError as e:
    print(f"Error esperado: {e}")
    print()
    # pandas tiene encoding_errors= para manejar esto
    df_fallback = pd.read_csv(
        io.BytesIO(csv_bytes_latin1),
        encoding="utf-8",
        encoding_errors="replace"  # reemplaza bytes inválidos con '?'
    )
    print("Con encoding_errors='replace':")
    print(df_fallback)

Error esperado: 'utf-8' codec can't decode byte 0xe9 in position 17: invalid continuation byte

Con encoding_errors='replace':
  nombre     ciudad
0   Jos�     M�rida
1  �ngel  Quer�taro
2   �o�o   Campeche


### Auto-detección de encoding con chardet

Cuando no sabes qué encoding tiene un archivo, `chardet` puede adivinar analizando los patrones de bytes. No es perfecto — da una probabilidad (confidence) — pero es el punto de partida.

In [22]:
# Detectar encoding de un archivo desconocido
# En producción leerías con 'rb' desde disco: open('archivo.csv', 'rb')
archivo_misterioso = csv_content.encode("latin-1")

resultado = chardet.detect(archivo_misterioso)
print(resultado)
# Ejemplo de salida: {'encoding': 'ISO-8859-1', 'confidence': 0.73, 'language': ''}

{'encoding': 'ISO-8859-1', 'confidence': 0.73, 'language': ''}


In [23]:
# Usar el encoding detectado automáticamente
encoding_detectado = resultado["encoding"]
confianza = resultado["confidence"]
print(f"Encoding detectado: {encoding_detectado} (confianza: {confianza:.0%})")

# En código real, podrías poner un umbral de confianza
if confianza > 0.7:
    df_auto = pd.read_csv(io.BytesIO(archivo_misterioso), encoding=encoding_detectado)
    print("\nDataFrame leído con encoding auto-detectado:")
    print(df_auto)
else:
    print("Confianza baja — revisar manualmente")

Encoding detectado: ISO-8859-1 (confianza: 73%)

DataFrame leído con encoding auto-detectado:
  nombre     ciudad
0   José     Mérida
1  Ángel  Querétaro
2   ñoño   Campeche


### charset_normalizer como alternativa

`charset_normalizer` es una alternativa más moderna a chardet, con mejor precisión en algunos casos. Ya viene incluida con `requests`.

In [24]:
# charset_normalizer tiene una interfaz diferente — devuelve un objeto Results
results = from_bytes(archivo_misterioso)

# best() da el match con mayor confidence
mejor = results.best()
print(f"Encoding detectado: {mejor.encoding}")
print(f"Confianza: {mejor.chaos:.3f}")
print(f"Texto decodificado: {str(mejor)[:50]}...")

Encoding detectado: cp1250
Confianza: 0.000
Texto decodificado: nombre,ciudad
José,Mérida
Ángel,Querétaro
ńońo,Cam...


### El BOM de UTF-8 (utf-8-sig)

Excel en Windows guarda archivos CSV con **BOM** (Byte Order Mark): 3 bytes invisibles al inicio del archivo (`\xef\xbb\xbf`). Si lees ese archivo como `utf-8` normal, el nombre de la primera columna tendrá esos bytes invisibles al inicio — un bug que puede ser muy difícil de encontrar.

In [25]:
# Simular un CSV guardado por Excel con BOM
BOM = b"\xef\xbb\xbf"  # los 3 bytes invisibles que Excel agrega
contenido_csv = b"nombre,edad,ciudad\nAna,28,CDMX\nLuis,34,GDL"
csv_con_bom = BOM + contenido_csv

print(f"Primeros 10 bytes del archivo: {csv_con_bom[:10]}")
print(f"BOM visible: {csv_con_bom[:3] == BOM}")

Primeros 10 bytes del archivo: b'\xef\xbb\xbfnombre,'
BOM visible: True


In [26]:
# El bug: leer como utf-8 normal
df_bom_mal = pd.read_csv(io.BytesIO(csv_con_bom), encoding="utf-8")
print("Columnas con utf-8:", df_bom_mal.columns.tolist())
print("¿Primera columna tiene BOM?", df_bom_mal.columns[0].startswith("\ufeff"))
print()

# Esto causa errores silenciosos: df['nombre'] funciona, df_bom_mal['nombre'] falla
try:
    _ = df_bom_mal["nombre"]
    print("Acceso a 'nombre' exitoso")
except KeyError:
    print("KeyError: 'nombre' no existe — la columna tiene el BOM en su nombre")

Columnas con utf-8: ['nombre', 'edad', 'ciudad']
¿Primera columna tiene BOM? False

Acceso a 'nombre' exitoso


In [27]:
# La solución: utf-8-sig elimina el BOM automáticamente
df_bom_ok = pd.read_csv(io.BytesIO(csv_con_bom), encoding="utf-8-sig")
print("Columnas con utf-8-sig:", df_bom_ok.columns.tolist())
print()
df_bom_ok

Columnas con utf-8-sig: ['nombre', 'edad', 'ciudad']



,nombre,edad,ciudad
0,Ana,28,CDMX
1,Luis,34,GDL


💡 **Prueba esto:** abre un archivo en Excel en Windows y guárdalo como CSV. Luego verifica sus primeros 3 bytes en Python — casi siempre verás el BOM.

### Ejercicio 2

Tienes un archivo con encoding desconocido. Completa el código para:
1. Detectar el encoding con chardet
2. Leer el CSV con el encoding correcto
3. Verificar que los acentos se ven bien

In [32]:
# Archivo "desconocido" — no sabes su encoding
archivo_ej2 = "país,población,región\nMéxico,130000000,América\nEspaña,47000000,Europa".encode("cp1252")

# 1. Detecta el encoding
deteccion = chardet.detect(archivo_ej2)
print(f"Encoding detectado: {deteccion['encoding']} (confianza: {deteccion['confidence']:.0%})")

# 2. Lee el CSV con el encoding detectado
df_ej2 = pd.read_csv(
    io.BytesIO(archivo_ej2),
    encoding= deteccion['encoding']
)

# 3. Verifica los acentos
print(df_ej2)

Encoding detectado: ISO-8859-1 (confianza: 73%)
     país  población   región
0  México  130000000  América
1  España   47000000   Europa


<details>
<summary>Ver solución</summary>

```python
deteccion = chardet.detect(archivo_ej2)
print(f"Encoding detectado: {deteccion['encoding']} (confianza: {deteccion['confidence']:.0%})")

df_ej2 = pd.read_csv(
    io.BytesIO(archivo_ej2),
    encoding=deteccion["encoding"]
)
print(df_ej2)
```
</details>

---

## Sección 3: JSON — del módulo nativo a pandas

### El módulo `json` nativo de Python

Antes de llegar a pandas, conviene saber qué hace Python con JSON directamente. `json.loads()` convierte un string JSON a objetos Python, `json.dumps()` hace lo inverso.

In [33]:
# json.loads: string → objeto Python
json_str = '{"nombre": "Ana", "edad": 28, "activa": true}'
objeto = json.loads(json_str)
print(type(objeto), objeto)
print(f"nombre: {objeto['nombre']}, tipo: {type(objeto['nombre'])}")

<class 'dict'> {'nombre': 'Ana', 'edad': 28, 'activa': True}
nombre: Ana, tipo: <class 'str'>


In [34]:
# json.dumps: objeto Python → string JSON
datos = {"usuarios": ["Ana", "Luis"], "total": 2, "activo": True}
json_formateado = json.dumps(datos, indent=2, ensure_ascii=False)
print(json_formateado)
# ensure_ascii=False es clave para que los acentos no se escapen como \u00e1

{
  "usuarios": [
    "Ana",
    "Luis"
  ],
  "total": 2,
  "activo": true
}


### JSON plano → DataFrame directo

Cuando el JSON es un array de objetos con la misma estructura, `pd.DataFrame()` lo convierte directamente.

In [35]:
# Array de objetos planos — la estructura más amigable con pandas
json_plano = [
    {"id": 1, "nombre": "Ana", "score": 8.5},
    {"id": 2, "nombre": "Luis", "score": 7.2},
    {"id": 3, "nombre": "María", "score": 9.1},
]

# pd.DataFrame() entiende directamente una lista de dicts
df_plano = pd.DataFrame(json_plano)
df_plano

,id,nombre,score
0,1,Ana,8.5
1,2,Luis,7.2
2,3,María,9.1


### JSON anidado — el caso real de APIs

Las APIs rara vez devuelven JSON plano. Normalmente hay estructuras anidadas: objetos dentro de objetos, arrays dentro de objetos. `pd.json_normalize()` es la función clave para aplanar estas estructuras.

In [36]:
# Estructura típica de respuesta de API: objeto raíz con array de registros
data_anidada = {
    "status": "ok",
    "total": 3,
    "usuarios": [
        {
            "id": 1,
            "nombre": "Ana",
            "direccion": {"ciudad": "CDMX", "cp": "06600", "estado": "CDMX"},
            "suscripcion": "premium"
        },
        {
            "id": 2,
            "nombre": "Luis",
            "direccion": {"ciudad": "Guadalajara", "cp": "44100", "estado": "JAL"},
            "suscripcion": "free"
        },
        {
            "id": 3,
            "nombre": "María",
            "direccion": {"ciudad": "Monterrey", "cp": "64000", "estado": "NL"},
            "suscripcion": "premium"
        },
    ]
}

In [37]:
# Sin json_normalize: el dict 'direccion' queda como un objeto en la columna
df_sin_normalizar = pd.DataFrame(data_anidada["usuarios"])
print("Sin normalizar:")
print(df_sin_normalizar)
print(f"\nTipo de 'direccion': {type(df_sin_normalizar['direccion'][0])}")

Sin normalizar:
   id nombre                                          direccion suscripcion
0   1    Ana  {'ciudad': 'CDMX', 'cp': '06600', 'estado': 'C...     premium
1   2   Luis  {'ciudad': 'Guadalajara', 'cp': '44100', 'esta...        free
2   3  María  {'ciudad': 'Monterrey', 'cp': '64000', 'estado...     premium

Tipo de 'direccion': <class 'dict'>


In [38]:
# json_normalize aplana la estructura anidada en columnas separadas
# record_path: dónde están los registros que quiero como filas
# meta: campos del nivel superior que quiero conservar como columnas
df_normalizado = pd.json_normalize(
    data_anidada,
    record_path="usuarios",
    meta=["status", "total"]  # campos del objeto raíz que queremos conservar
)
df_normalizado

,id,nombre,suscripcion,direccion.ciudad,direccion.cp,direccion.estado,status,total
0,1,Ana,premium,CDMX,06600,CDMX,ok,3
1,2,Luis,free,Guadalajara,44100,JAL,ok,3
2,3,María,premium,Monterrey,64000,NL,ok,3


In [39]:
# Nota: json_normalize usa '.' como separador para campos anidados
# direccion.ciudad, direccion.cp, direccion.estado
# Puedes cambiarlo con sep=
df_sep = pd.json_normalize(
    data_anidada["usuarios"],  # directamente el array de usuarios
    sep="_"  # ahora serán: direccion_ciudad, direccion_cp, etc.
)
print("Columnas con sep='_':")
print(df_sep.columns.tolist())
df_sep.head(2)

Columnas con sep='_':
['id', 'nombre', 'suscripcion', 'direccion_ciudad', 'direccion_cp', 'direccion_estado']


,id,nombre,suscripcion,direccion_ciudad,direccion_cp,direccion_estado
0,1,Ana,premium,CDMX,06600,CDMX
1,2,Luis,free,Guadalajara,44100,JAL


💡 **Prueba esto:** agrega un campo extra al primer usuario que no existe en los demás, por ejemplo `"verified": true`. ¿Qué hace `json_normalize` con los valores faltantes en los otros registros?

### `pd.read_json()` — cuándo usarlo

`pd.read_json()` es conveniente para JSON con formato estándar (arrays de objetos). El parámetro `orient=` le dice a pandas cómo está estructurado el JSON.

In [40]:
# orient='records': [{col1: val, col2: val}, ...] — el más común de APIs
json_records = json.dumps([
    {"nombre": "Ana", "score": 8.5},
    {"nombre": "Luis", "score": 7.2}
])
df_records = pd.read_json(io.StringIO(json_records), orient="records")
print("orient='records':")
print(df_records)

orient='records':
  nombre  score
0    Ana    8.5
1   Luis    7.2


In [41]:
# orient='split': {"columns": [...], "index": [...], "data": [[...]]}
# — compacto, útil para transferir DataFrames
json_split = json.dumps({
    "columns": ["nombre", "score"],
    "index": [0, 1],
    "data": [["Ana", 8.5], ["Luis", 7.2]]
})
df_split = pd.read_json(io.StringIO(json_split), orient="split")
print("orient='split':")
print(df_split)

orient='split':
  nombre  score
0    Ana    8.5
1   Luis    7.2


In [42]:
# orient='columns': {col: {index: value}} — el default de pandas to_json()
json_columns = json.dumps({
    "nombre": {"0": "Ana", "1": "Luis"},
    "score": {"0": 8.5, "1": 7.2}
})
df_columns = pd.read_json(io.StringIO(json_columns), orient="columns")
print("orient='columns':")
print(df_columns)

orient='columns':
  nombre  score
0    Ana    8.5
1   Luis    7.2


### Manejo de errores en JSON

Los JSON de APIs a veces tienen campos faltantes o tipos inconsistentes. Vale la pena manejar estos casos explícitamente.

In [43]:
# JSON con estructura inconsistente — campo que a veces es string, a veces int
json_inconsistente = [
    {"id": 1, "valor": "100"},
    {"id": 2, "valor": 200},      # int en vez de string
    {"id": 3},                     # campo 'valor' ausente
    {"id": 4, "valor": None},      # null explícito
]

# pd.DataFrame maneja estos casos con NaN para valores ausentes
df_incons = pd.DataFrame(json_inconsistente)
print(df_incons)
print("\nTipos:", df_incons.dtypes.to_dict())

   id valor
0   1   100
1   2   200
2   3   NaN
3   4  None

Tipos: {'id': dtype('int64'), 'valor': dtype('O')}


In [44]:
# Para JSON malformado, try/except es más explícito que errors='ignore'
json_malo = '{"nombre": "Ana", "edad": }'

try:
    datos = json.loads(json_malo)
    df_malo = pd.DataFrame([datos])
except json.JSONDecodeError as e:
    print(f"JSON malformado en posición {e.pos}: {e.msg}")
    print("Contexto:", json_malo[max(0, e.pos-10):e.pos+10])

JSON malformado en posición 26: Expecting value
Contexto: , "edad": }


### Ejercicio 3

Dada la siguiente estructura JSON de una API de pedidos, normalízala en un DataFrame donde cada fila sea un producto, conservando `id_pedido` y `fecha` como columnas adicionales.

In [45]:
pedidos_api = {
    "pedidos": [
        {
            "id_pedido": "P001",
            "fecha": "2026-03-01",
            "productos": [
                {"sku": "LAP-001", "nombre": "Laptop", "cantidad": 1, "precio": 22000},
                {"sku": "MOU-001", "nombre": "Mouse", "cantidad": 2, "precio": 350},
            ]
        },
        {
            "id_pedido": "P002",
            "fecha": "2026-03-02",
            "productos": [
                {"sku": "MON-001", "nombre": "Monitor", "cantidad": 1, "precio": 8500},
            ]
        }
    ]
}

# Tu solución: usa pd.json_normalize() con record_path y meta
df_ej3 = pd.DataFrame()
df_ej3

""


<details>
<summary>Ver solución</summary>

```python
df_ej3 = pd.json_normalize(
    pedidos_api,
    record_path=["pedidos", "productos"],
    meta=[["pedidos", "id_pedido"], ["pedidos", "fecha"]]
)
df_ej3
```
</details>

---

## Sección 4: Parquet — acceso rápido a columnas

Parquet es un formato columnar binario: eficiente en almacenamiento y muy rápido para leer columnas específicas. Ya vimos su comparación con CSV en el módulo 12. Aquí solo el workflow esencial.

In [46]:
# Crear un DataFrame de ejemplo para guardar
import tempfile, os

df_origen = pd.DataFrame({
    "id": range(1000),
    "nombre": [f"usuario_{i}" for i in range(1000)],
    "score": np.random.uniform(0, 10, 1000),
    "categoria": np.random.choice(["A", "B", "C"], 1000),
    "fecha": pd.date_range("2026-01-01", periods=1000, freq="h")
})

# Guardar como Parquet con pyarrow como engine
ruta_parquet = os.path.join(tempfile.gettempdir(), "ejemplo.parquet")
df_origen.to_parquet(ruta_parquet, engine="pyarrow", index=False)
print(f"Guardado en: {ruta_parquet}")
print(f"Tamaño: {os.path.getsize(ruta_parquet):,} bytes")

Guardado en: /tmp/ejemplo.parquet
Tamaño: 33,874 bytes


In [47]:
# La ventaja de Parquet: leer solo las columnas que necesitas
# En un archivo con millones de filas, esto es órdenes de magnitud más rápido
df_columnas = pd.read_parquet(
    ruta_parquet,
    engine="pyarrow",
    columns=["id", "score", "categoria"]  # solo estas columnas se leen del disco
)
print(f"Forma: {df_columnas.shape}")
print(f"Columnas: {df_columnas.columns.tolist()}")
df_columnas.head()

Forma: (1000, 3)
Columnas: ['id', 'score', 'categoria']


,id,score,categoria
0,0,4.167284,B
1,1,6.938450,A
2,2,8.357739,C
3,3,3.248484,B
4,4,0.271326,A


In [48]:
# Parquet preserva los tipos — no hay conversión de fechas ni de categorías
print("Tipos originales:", df_origen.dtypes.to_dict())
print()
df_todo = pd.read_parquet(ruta_parquet)
print("Tipos leídos:", df_todo.dtypes.to_dict())

# Limpiar el archivo temporal
os.remove(ruta_parquet)

Tipos originales: {'id': dtype('int64'), 'nombre': dtype('O'), 'score': dtype('float64'), 'categoria': dtype('O'), 'fecha': dtype('<M8[ns]')}

Tipos leídos: {'id': dtype('int64'), 'nombre': dtype('O'), 'score': dtype('float64'), 'categoria': dtype('O'), 'fecha': dtype('<M8[ns]')}


---

## Sección 5: Ejercicio integrador

Este ejercicio combina todo lo visto: CSV con encoding especial, detección automática, corrección de mojibake y normalización de JSON anidado.

### Parte 1: CSV con separador `;` y encoding latin-1

In [49]:
# Simular un CSV exportado desde un sistema legacy en México
csv_legado = (
    "clave;municipio;población;estado\n"
    "01001;Aguascalientes;979884;AGS\n"
    "02001;Ensenada;536091;BC\n"
    "09002;Azcapotzalco;400161;CDMX\n"
    "14039;Guadalajara;1385629;JAL\n"
    "19039;Monterrey;1135512;NL"
)
archivo_legado = csv_legado.encode("latin-1")

# Parte 1a: Lee el CSV con los parámetros correctos
df_municipios = pd.read_csv(
    io.BytesIO(archivo_legado),
    sep=";",
    encoding="latin-1",
    dtype={"clave": str}  # mantener cero inicial
)
print("CSV leído correctamente:")
df_municipios

CSV leído correctamente:


,clave,municipio,población,estado
0,01001,Aguascalientes,979884,AGS
1,02001,Ensenada,536091,BC
2,09002,Azcapotzalco,400161,CDMX
3,14039,Guadalajara,1385629,JAL
4,19039,Monterrey,1135512,NL


### Parte 2: Detección automática de encoding

In [50]:
# Simular que no sabemos el encoding del archivo
def leer_csv_encoding_desconocido(contenido_bytes):
    """Lee un CSV detectando automáticamente su encoding."""
    # Detectar encoding
    deteccion = chardet.detect(contenido_bytes)
    encoding = deteccion["encoding"]
    confianza = deteccion["confidence"]

    print(f"Encoding detectado: {encoding} (confianza: {confianza:.0%})")

    # Leer con el encoding detectado, con fallback a latin-1
    try:
        return pd.read_csv(io.BytesIO(contenido_bytes), sep=";", encoding=encoding, dtype={"clave": str})
    except (UnicodeDecodeError, LookupError):
        print("Encoding detectado falló, usando latin-1 como fallback")
        return pd.read_csv(io.BytesIO(contenido_bytes), sep=";", encoding="latin-1", dtype={"clave": str})

df_auto = leer_csv_encoding_desconocido(archivo_legado)
df_auto

Encoding detectado: ISO-8859-1 (confianza: 73%)


,clave,municipio,población,estado
0,01001,Aguascalientes,979884,AGS
1,02001,Ensenada,536091,BC
2,09002,Azcapotzalco,400161,CDMX
3,14039,Guadalajara,1385629,JAL
4,19039,Monterrey,1135512,NL


### Parte 3: Corrección de mojibake

In [51]:
# Simular texto con mojibake: se guardó en latin-1 pero alguien lo leyó como cp500
texto_con_acento = "Población de Jalisco: güero y ñoño"
bytes_originales = texto_con_acento.encode("latin-1")

# Mojibake: decodificado con el encoding equivocado
texto_mojibake = bytes_originales.decode("cp500", errors="replace")
print("Con mojibake:", texto_mojibake)

Con mojibake: &?Â%/ÄÑ3>ÀÁ[/%ÑËÄ?ÅÜÁÊ?`1?1?


In [52]:
# Estrategia de corrección cuando conoces el encoding original
# 1. Re-encodear a bytes con el encoding en que fue leído erróneamente
# 2. Decodificar con el encoding correcto
def corregir_mojibake(texto_mal, encoding_incorrecto, encoding_correcto):
    """Intenta revertir un mojibake conociendo los dos encodings involucrados."""
    try:
        bytes_recuperados = texto_mal.encode(encoding_incorrecto, errors="replace")
        return bytes_recuperados.decode(encoding_correcto, errors="replace")
    except (UnicodeEncodeError, UnicodeDecodeError) as e:
        return f"No se pudo corregir: {e}"

# Alternativa más práctica: si sabes los bytes originales, decodificar directamente
texto_correcto = bytes_originales.decode("latin-1")
print("Texto corregido:", texto_correcto)
print("¿Igual al original?", texto_correcto == texto_con_acento)

Texto corregido: Población de Jalisco: güero y ñoño
¿Igual al original? True


### Parte 4: Normalizar JSON anidado de una API

In [53]:
# Respuesta típica de una API de datos municipales
api_respuesta = {
    "version": "2.1",
    "fuente": "INEGI",
    "registros": [
        {
            "clave": "14039",
            "nombre": "Guadalajara",
            "ubicacion": {"estado": "Jalisco", "region": "Occidente", "lat": 20.67, "lon": -103.35},
            "indicadores": {"poblacion": 1385629, "densidad": 9766, "area_km2": 142}
        },
        {
            "clave": "19039",
            "nombre": "Monterrey",
            "ubicacion": {"estado": "Nuevo León", "region": "Noreste", "lat": 25.67, "lon": -100.31},
            "indicadores": {"poblacion": 1135512, "densidad": 6847, "area_km2": 166}
        },
        {
            "clave": "09002",
            "nombre": "Azcapotzalco",
            "ubicacion": {"estado": "CDMX", "region": "Centro", "lat": 19.49, "lon": -99.18},
            "indicadores": {"poblacion": 400161, "densidad": 14720, "area_km2": 27}
        }
    ]
}

# Normalizar: cada registro como fila, campos anidados como columnas
df_api = pd.json_normalize(
    api_respuesta,
    record_path="registros",
    meta=["fuente"],
    sep="_"  # ubicacion_estado, indicadores_poblacion, etc.
)

print("Columnas:", df_api.columns.tolist())
print()
df_api

Columnas: ['clave', 'nombre', 'ubicacion_estado', 'ubicacion_region', 'ubicacion_lat', 'ubicacion_lon', 'indicadores_poblacion', 'indicadores_densidad', 'indicadores_area_km2', 'fuente']



,clave,nombre,ubicacion_estado,ubicacion_region,ubicacion_lat,ubicacion_lon,indicadores_poblacion,indicadores_densidad,indicadores_area_km2,fuente
0,14039,Guadalajara,Jalisco,Occidente,20.67,-103.35,1385629,9766,142,INEGI
1,19039,Monterrey,Nuevo León,Noreste,25.67,-100.31,1135512,6847,166,INEGI
2,09002,Azcapotzalco,CDMX,Centro,19.49,-99.18,400161,14720,27,INEGI


In [54]:
# Combinar todo: unir el CSV de municipios con la respuesta de API por clave
# Primero aseguramos que clave es string en ambos
df_api["clave"] = df_api["clave"].astype(str)
df_municipios["clave"] = df_municipios["clave"].astype(str)

df_combinado = df_municipios.merge(
    df_api[["clave", "ubicacion_region", "indicadores_densidad"]],
    on="clave",
    how="left"  # left join para no perder municipios sin datos de API
)

df_combinado

,clave,municipio,población,estado,ubicacion_region,indicadores_densidad
0,01001,Aguascalientes,979884,AGS,NaN,NaN
1,02001,Ensenada,536091,BC,NaN,NaN
2,09002,Azcapotzalco,400161,CDMX,Centro,14720.0
3,14039,Guadalajara,1385629,JAL,Occidente,9766.0
4,19039,Monterrey,1135512,NL,Noreste,6847.0


---

## Resumen

| Situación | Solución |
|---|---|
| CSV con `;` como separador | `sep=";"` |
| Archivo sin header | `header=None, names=[...]` |
| Metadata antes de los datos | `skiprows=N` |
| Solo algunas columnas | `usecols=["a", "b"]` |
| Tipos incorrectos / códigos postales | `dtype={"col": str}` |
| Valores nulos no estándar | `na_values=["." , "ND", "99999"]` |
| Encoding desconocido | `chardet.detect(bytes)` |
| Archivo de Excel Windows | `encoding="utf-8-sig"` |
| JSON plano de API | `pd.DataFrame(lista_de_dicts)` |
| JSON anidado | `pd.json_normalize(data, record_path=..., meta=...)` |
| Lectura eficiente de columnas | Parquet + `pd.read_parquet(columns=[...])` |

**Regla de oro:** cuando un archivo no se lee bien, revisa primero el separador y el encoding — el 80% de los problemas de IO vienen de ahí.